In [3]:
# src/features.py
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from pathlib import Path

# --- Paths ---
PROJECT_ROOT = Path.home() / "Downloads/epidemic-project"
DATA_CURATED = PROJECT_ROOT / "data_curated"
OUT_DIR = DATA_CURATED
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Load curated panel ---
panel = pd.read_csv(DATA_CURATED / "panel_state_daily.csv", parse_dates=['Date'])
panel = panel.sort_values(['state','Date'])

# --- Compute daily vaccination (if only cumulative is available) ---
if 'total_vaccinated' in panel.columns:
    panel['daily_vax'] = panel.groupby('state')['total_vaccinated'].diff().fillna(0)
else:
    panel['daily_vax'] = 0

# --- Compute tests per day if present ---
if 'TotalSamples' in panel.columns:
    panel['daily_tests'] = panel.groupby('state')['TotalSamples'].diff().fillna(0)
else:
    panel['daily_tests'] = 0

# --- Feature engineering ---
panel['weekday'] = panel['Date'].dt.weekday
panel['vax_rate'] = panel['total_vaccinated'] / panel['population']
panel['new_confirmed'] = panel['new_confirmed'].clip(lower=0)

# --- Label encode states to indices ---
le = LabelEncoder()
panel['state_id'] = le.fit_transform(panel['state'])

# --- Normalize numeric features within each state ---
num_features = ['active','new_confirmed','daily_tests','daily_vax','vax_rate']
scalers = {f: MinMaxScaler() for f in num_features}

for f in num_features:
    # fill missing values within each state and keep index aligned
    panel[f] = panel.groupby('state')[f].transform(lambda x: x.fillna(0))
    # scale
    panel[f] = scalers[f].fit_transform(panel[[f]].values)

# --- Save the processed panel ---
panel.to_csv(OUT_DIR / "panel_for_model.csv", index=False)
print(f"Saved features to {OUT_DIR / 'panel_for_model.csv'}")


Saved features to C:\Users\dedee\Downloads\epidemic-project\data_curated\panel_for_model.csv
